# EDA de SALUD-001: establecimientos de salud (CLUES)

Exploración de la hoja `CLUES_202608` de `data/raw/ESTABLECIMIENTO_SALUD_202608.xlsx`, el catálogo de establecimientos de salud de la Secretaría de Salud. Solo lectura y conteos: sin cambios al archivo y sin limpieza de datos. Lo que salga de aquí sirve para el diccionario de datos y para las reglas de limpieza.

- Fuente: Secretaría de Salud, gobi.salud.gob.mx
- Descarga: https://gobi.salud.gob.mx/gobi/catalogos/catalogosmaestros/ESTABLECIMIENTO_SALUD_202608.xlsx?v=2026.09.23
- Fecha de descarga: 07/10/2026

## 0. Preparación

Carga de librerías y de las rutas que ya define `health_data.pipeline`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from health_data.pipeline import RAW_DATA_DIR  # noqa: E402

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

Lectura de la hoja tal como viene. 

In [2]:
ARCHIVO = RAW_DATA_DIR / "ESTABLECIMIENTO_SALUD_202608.xlsx"
df = pd.read_excel(ARCHIVO, sheet_name="CLUES_202608")
df.shape

(64450, 68)

Separación de los registros de Sonora (clave de entidad 26) para las siguientes secciones.

In [3]:
son = df[df["CLAVE DE LA ENTIDAD"] == 26]
len(son)

1683

Colores y funciones para las gráficas del notebook; en las barras, los vacíos aparecen como "(sin dato)".

In [4]:
import plotly.express as px

AZUL = "#2a78d6"
COLORES_ESTATUS = {
    "EN OPERACION": "#2a78d6",
    "FUERA DE OPERACION": "#eb6834",
    "PENDIENTE DE ENTRAR EN OPERACION": "#1baf7a",
}
PALABRAS_VACIAS = set(
    "de la el en y a los las del se por con que al es su para un una lo no le o e "
    "sus fue ser esta este como".split()
)


def conteo(serie):
    return serie.fillna("(sin dato)").value_counts()


def palabras_por_registro(serie):
    palabras = serie.dropna().str.lower().str.findall(r"[a-záéíóúñü]+").apply(set).explode()
    palabras = palabras[~palabras.isin(PALABRAS_VACIAS) & (palabras.str.len() > 2)]
    return palabras.value_counts()


def barras(conteos, titulo, eje_x="Registros"):
    conteos = conteos.sort_values()
    fig = px.bar(
        x=conteos.values, y=conteos.index, orientation="h", text=conteos.values, title=titulo
    )
    fig.update_traces(marker_color=AZUL, textposition="outside", cliponaxis=False)
    fig.update_layout(
        template="plotly_white",
        height=max(260, 22 * len(conteos) + 120),
        xaxis_title=eje_x,
        yaxis_title=None,
        barcornerradius=4,
    )
    fig.show()


def caja(col, titulo):
    fig = px.box(son, y=col, points="outliers", hover_data=["CLUES", "MUNICIPIO"], title=titulo)
    fig.update_traces(marker_color=AZUL)
    fig.update_layout(template="plotly_white", height=420, yaxis_title=None)
    fig.show()

## 1. Estructura

Resumen de las 68 columnas: tipo y número de valores de cada una.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64450 entries, 0 to 64449
Data columns (total 68 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   CLUES                                       64450 non-null  object 
 1   CLAVE DE LA INSTITUCION                     64450 non-null  object 
 2   NOMBRE DE LA INSTITUCION                    64450 non-null  object 
 3   CLAVE DE LA ENTIDAD                         64450 non-null  int64  
 4   ENTIDAD                                     64450 non-null  object 
 5   CLAVE DEL MUNICIPIO                         64450 non-null  int64  
 6   MUNICIPIO                                   64450 non-null  object 
 7   CLAVE DE LA LOCALIDAD                       64450 non-null  int64  
 8   LOCALIDAD                                   64450 non-null  object 
 9   CLAVE DE LA JURISDICCION                    64450 non-null  int64  
 10  JURISDICCI

Formato de las claves geográficas y del código postal en algunos registros de Sonora.

In [6]:
son[
    [
        "CLUES",
        "CLAVE DE LA ENTIDAD",
        "CLAVE DEL MUNICIPIO",
        "MUNICIPIO",
        "CLAVE DE LA LOCALIDAD",
        "LOCALIDAD",
        "CODIGO POSTAL",
    ]
].head()

,CLUES,CLAVE DE LA ENTIDAD,CLAVE DEL MUNICIPIO,MUNICIPIO,CLAVE DE LA LOCALIDAD,LOCALIDAD,CODIGO POSTAL
51151,SRCIJ000013,26,43,NOGALES,499,SANTA CLARA,84085.0
51152,SRCIJ000025,26,30,HERMOSILLO,1,HERMOSILLO,83040.0
51153,SRDIF000011,26,30,HERMOSILLO,1,HERMOSILLO,83140.0
51154,SRHUN000013,26,12,BACUM,86,LOMA DE BACUM,99999.0
51155,SRIMB000016,26,29,GUAYMAS,325,VICAM (SWITCH),85510.0


Formato de las columnas de fecha.

In [7]:
son.filter(like="FECHA").head()

,FECHA DE CONSTRUCCION,FECHA DE INICIO DE OPERACION,FECHA ULTIMO MOVIMIENTO,FECHA EFECTIVA DE BAJA
51151,NaN,16/11/2006,16/10/2018,NaN
51152,15/01/1976,15/01/1976,30/06/2026,NaN
51153,06/05/1988,10/11/1988,02/07/2015,NaN
51154,15/08/2012,30/10/2012,21/01/2021,07/01/2021
51155,05/06/2024,05/06/2024,09/10/2024,NaN


Hallazgos:
- 64,450 establecimientos de todo el país y 68 columnas.
- Claves de entidad, municipio y localidad como números, sin ceros a la izquierda.
- Código postal como decimal y fechas como texto dd/mm/aaaa.

## 2. Valores faltantes

Porcentaje de nulos por columna en el país y en Sonora (solo columnas con algún nulo en Sonora).

In [8]:
nulos = pd.DataFrame(
    {
        "%_pais": df.isna().mean().mul(100).round(1),
        "%_sonora": son.isna().mean().mul(100).round(1),
    }
)
nulos[nulos["%_sonora"] > 0].sort_values("%_sonora", ascending=False)

,%_pais,%_sonora
UNIDAD MOVIL MARCA ESPECIFICA,98.3,99.9
EXTENSION TELEFONICA 2 DEL ESTABLECIMIENTO,99.0,98.4
UNIDAD MOVIL TIPOLOGIA,95.1,98.3
CLAVE UNIDAD MOVIL TIPOLOGIA,95.1,98.3
UNIDAD MOVIL MODELO,92.0,98.2
CLAVE UNIDAD MOVIL MARCA,92.2,98.2
UNIDAD MOVIL MARCA,92.2,98.2
UNIDAD MOVIL PROGRAMA,93.0,96.9
CLAVE UNIDAD MOVIL PROGRAMA,93.0,96.9
UNIDAD MOVIL TIPO,92.0,96.9


Conteo de textos vacíos o con solo espacios, que pandas no marca como nulos.

In [9]:
vacios = son.select_dtypes("object").apply(lambda s: s.str.strip().eq("").sum())
vacios[vacios > 0]

ASENTAMIENTO    110
dtype: int64

Conteo de valores que en la práctica significan "sin dato".

In [10]:
marcadores = {
    "NUMERO EXTERIOR": "SIN NÚMERO",
    "VIALIDAD": "SIN NOMBRE",
    "CODIGO POSTAL": 99999,
    "NOMBRE DE TIPOLOGIA": "NO ESPECIFICADO",
    "NOMBRE DE LA INS ADM": "NO ESPECIFICADO",
}
pd.Series({f"{col} = {valor}": son[col].eq(valor).sum() for col, valor in marcadores.items()})

NUMERO EXTERIOR = SIN NÚMERO              1243
VIALIDAD = SIN NOMBRE                      287
CODIGO POSTAL = 99999                      245
NOMBRE DE TIPOLOGIA = NO ESPECIFICADO      140
NOMBRE DE LA INS ADM = NO ESPECIFICADO     109
dtype: int64

Conteo de valores con espacios al inicio o al final, que pueden partir una categoría en dos.

In [11]:
def con_espacios(s):
    limpio = s.str.strip()
    return (limpio.notna() & s.ne(limpio)).sum()


espacios = son.select_dtypes("object").apply(con_espacios)
espacios[espacios > 0].sort_values(ascending=False)

NOMBRE DE LA INS ADM         555
NOMBRE DE LA INSTITUCION     290
ASENTAMIENTO                 171
VIALIDAD                     148
NUMERO EXTERIOR              137
REFERENCIAS DEL DOMICILIO      8
NOMBRE DE TIPOLOGIA            6
NOMBRE DE LA UNIDAD            2
OBSERVACIONES AL REGISTRO      2
NUMERO INTERIOR                1
dtype: int64

Hallazgos:
- Sin nulos en las columnas clave (CLUES, institución, nivel, estatus, municipio y localidad); en Sonora faltan coordenadas en el 1.7 %.
- Faltantes escondidos: 1243 "SIN NÚMERO", 287 "SIN NOMBRE", 245 códigos postales 99999 y 110 asentamientos en blanco.
- Categorías con espacios de más, como "SERVICIOS DE SALUD IMSS BIENESTAR ".

## 3. CLUES repetidas

Conteo de CLUES que aparecen más de una vez en toda la hoja.

In [12]:
df["CLUES"].duplicated().sum()

np.int64(0)

Formato de las CLUES: 5 letras y 6 dígitos.

In [13]:
df["CLUES"].str.fullmatch(r"[A-Z]{5}\d{6}").value_counts()

CLUES
True    64450
Name: count, dtype: int64

Prefijo de las CLUES de Sonora (debe ser SR).

In [14]:
son["CLUES"].str[:2].value_counts()

CLUES
SR    1683
Name: count, dtype: int64

Hallazgos:
- Sin CLUES repetidas: la CLUES sirve como llave.
- Todas con formato de 5 letras y 6 dígitos; en Sonora todas empiezan con SR.

## 4. Estatus de operación

Conteo de establecimientos de Sonora por estatus.

In [15]:
son["ESTATUS DE OPERACION"].value_counts()

ESTATUS DE OPERACION
EN OPERACION                        943
FUERA DE OPERACION                  736
PENDIENTE DE ENTRAR EN OPERACION      4
Name: count, dtype: int64

Frecuencia por estatus de operación.

In [16]:
barras(conteo(son["ESTATUS DE OPERACION"]), "Registros por estatus de operación")

Hallazgo: 943 en operación, 736 fuera y 4 pendientes; el 44 % de los registros no está en operación.

Comparación con las cifras de referencia: 1678 en total y 939 en operación.

In [17]:
REF_TOTAL, REF_OPERACION = 1678, 939

pd.DataFrame(
    {
        "archivo": [len(son), son["ESTATUS DE OPERACION"].eq("EN OPERACION").sum()],
        "referencia": [REF_TOTAL, REF_OPERACION],
    },
    index=["total", "en operación"],
).assign(diferencia=lambda t: t["archivo"] - t["referencia"])

,archivo,referencia,diferencia
total,1683,1678,5
en operación,943,939,4


Altas más recientes (desde septiembre de 2026), que podrían explicar la diferencia.

In [18]:
fecha_mov = pd.to_datetime(son["FECHA ULTIMO MOVIMIENTO"], format="%d/%m/%Y")
altas_sep = son[son["ULTIMO MOVIMIENTO"].eq("ALTA") & (fecha_mov >= "2026-09-01")]
altas_sep[["CLUES", "NOMBRE DE LA INSTITUCION", "ESTATUS DE OPERACION", "FECHA ULTIMO MOVIMIENTO"]]

,CLUES,NOMBRE DE LA INSTITUCION,ESTATUS DE OPERACION,FECHA ULTIMO MOVIMIENTO
51444,SRIMB002903,SERVICIOS DE SALUD IMSS BIENESTAR,PENDIENTE DE ENTRAR EN OPERACION,10/09/2026
52060,SRSMP003913,SERVICIOS MEDICOS PRIVADOS,EN OPERACION,08/09/2026
52061,SRSMP003925,SERVICIOS MEDICOS PRIVADOS,EN OPERACION,08/09/2026
52062,SRSMP003930,SERVICIOS MEDICOS PRIVADOS,EN OPERACION,09/09/2026
52063,SRSMP003942,SERVICIOS MEDICOS PRIVADOS,EN OPERACION,09/09/2026


Fecha del último movimiento en todo el archivo.

In [19]:
pd.to_datetime(df["FECHA ULTIMO MOVIMIENTO"], format="%d/%m/%Y").max()

Timestamp('2026-09-22 00:00:00')

Frecuencia por último movimiento (alta, modificación o baja).

In [20]:
barras(conteo(son["ULTIMO MOVIMIENTO"]), "Registros por último movimiento")

Hallazgo: 649 altas, 605 modificaciones y 429 bajas; como hay 736 fuera de operación, 307 de ellos tienen como último movimiento una alta o una modificación.

Motivos de baja de los establecimientos fuera de operación, incluidos los que no traen motivo.

In [21]:
fuera = son[son["ESTATUS DE OPERACION"] == "FUERA DE OPERACION"]
barras(conteo(fuera["MOTIVO BAJA"]), "Motivo de baja (fuera de operación)")

Hallazgo: 198 de los 736 fuera de operación no tienen motivo; el más común es cambio de institución (268) y 9 se dieron de baja por CLUES duplicada.

Resumen:
- 5 más en total y 4 más en operación que la referencia; coincide con 5 altas del 8 al 10 de septiembre de 2026, así que la referencia parece ser de un corte anterior.
- El archivo dice 202608, pero trae movimientos hasta el 22/09/2026.
- Casi todos los fuera de operación tienen registrada su baja, ya sea con motivo o como último movimiento; solo 2 no tienen ninguno de los dos.

## 5. Establecimientos en operación

Niveles de atención, en la columna NIVEL ATENCION (su clave está en CLAVE NIVEL ATENCION):

- Primer nivel (1): consulta externa básica y prevención, como centros de salud, unidades de medicina familiar, casas de salud y consultorios.
- Segundo nivel (2): hospitales generales y clínicas de especialidades.
- Tercer nivel (3): hospitales de alta especialidad.
- No aplica (6): establecimientos sin atención médica directa, como oficinas, almacenes, laboratorios y asistencia social.

Solo establecimientos en operación, que son los que cuentan para medir acceso. El resto del notebook usa todos los registros de Sonora.

In [22]:
op = son[son["ESTATUS DE OPERACION"] == "EN OPERACION"]
len(op)

943

Cruce de institución y nivel de atención.

In [23]:
pd.crosstab(
    op["NOMBRE DE LA INSTITUCION"], op["NIVEL ATENCION"], margins=True, margins_name="Total"
).sort_values("Total", ascending=False)

NIVEL ATENCION,NO APLICA,PRIMER NIVEL,SEGUNDO NIVEL,TERCER NIVEL,Total
NOMBRE DE LA INSTITUCION,,,,,
Total,57,777,103,6,943
SERVICIOS MEDICOS PRIVADOS,9,292,49,2,352
SERVICIOS DE SALUD IMSS BIENESTAR,7,252,24,2,285
SECRETARIA DE SALUD,31,100,0,0,131
INSTITUTO MEXICANO DEL SEGURO SOCIAL,7,62,19,1,89
SERVICIOS MEDICOS ESTATALES,2,36,3,1,42
INSTITUTO DE SEGURIDAD Y SERVICIOS SOCIALES DE LOS TRABAJADORES DEL ESTADO,1,29,6,0,36
CENTROS DE INTEGRACION JUVENIL,0,2,0,0,2
SECRETARIA DE MARINA,0,1,1,0,2


Frecuencia por institución.

In [24]:
barras(conteo(op["NOMBRE DE LA INSTITUCION"]), "Establecimientos en operación por institución")

Hallazgo: privados (352) e IMSS-Bienestar (285) suman dos tercios de los establecimientos en operación; 6 instituciones tienen 1 o 2 y solo una escribe su nombre con acentos.

Cruce de tipo de establecimiento y nivel de atención.

In [25]:
pd.crosstab(
    op["NOMBRE TIPO ESTABLECIMIENTO"], op["NIVEL ATENCION"], margins=True, margins_name="Total"
)

NIVEL ATENCION,NO APLICA,PRIMER NIVEL,SEGUNDO NIVEL,TERCER NIVEL,Total
NOMBRE TIPO ESTABLECIMIENTO,,,,,
DE APOYO,52,0,0,0,52
DE ASISTENCIA SOCIAL,5,0,0,0,5
DE CONSULTA EXTERNA,0,777,8,0,785
DE HOSPITALIZACIÓN,0,0,95,6,101
Total,57,777,103,6,943


Frecuencia por tipo de establecimiento.

In [26]:
barras(conteo(op["NOMBRE TIPO ESTABLECIMIENTO"]), "Establecimientos en operación por tipo")

Hallazgo: 785 de 943 son de consulta externa y 101 de hospitalización; no hay vacíos.

Las 15 tipologías más comunes.

In [27]:
op["NOMBRE DE TIPOLOGIA"].value_counts().head(15)

NOMBRE DE TIPOLOGIA
CONSULTORIO ADYACENTE A FARMACIA             277
RURAL DE 01 NÚCLEO BÁSICO                    112
CASA DE SALUD                                 91
NO ESPECIFICADO                               85
UNIDAD DE MEDICINA FAMILIAR                   75
URBANO DE 01 NÚCLEOS BÁSICOS                  65
URBANO DE 02 NÚCLEOS BÁSICOS                  28
UNIDAD DE ESPECIALIDADES MÉDICAS (UNEMES)     26
OFICINAS ADMINISTRATIVAS                      19
UNIDAD MÓVIL                                  14
HOSPITAL GENERAL                              13
ALMACENES                                     12
URBANO DE 03 NÚCLEOS BÁSICOS                  12
CONSULTORIO DE ATENCION FAMILIAR              11
LABORATORIOS                                  10
Name: count, dtype: int64

Frecuencia por tipología, con todas las categorías.

In [28]:
barras(conteo(op["NOMBRE DE TIPOLOGIA"]), "Establecimientos en operación por tipología")

Hallazgo: hay 49 tipologías y 12 tienen un solo establecimiento; "NO ESPECIFICADO" es la cuarta más común (85) y los acentos no son uniformes (CLINICA junto a CLÍNICA HOSPITAL).

Tipologías del segundo nivel.

In [29]:
op.loc[op["NIVEL ATENCION"] == "SEGUNDO NIVEL", "NOMBRE DE TIPOLOGIA"].value_counts()

NOMBRE DE TIPOLOGIA
NO ESPECIFICADO                                      52
HOSPITAL GENERAL                                     12
HOSPITAL GENERAL DE ZONA                              8
HOSPITAL INTEGRAL (COMUNITARIO)                       6
CLÍNICA HOSPITAL                                      5
HOSPITAL GENERAL DE SUBZONA CON MEDICINA FAMILIAR     5
CLÍNICA DE ESPECIALIDADES                             4
UNIDAD DE ESPECIALIDADES MÉDICAS (UNEMES)             2
HOSPITAL PSIQUIÁTRICO                                 2
HOSPITAL GENERAL DE SUBZONA                           2
HOSPITAL ESPECIALIZADO                                1
UNIDAD DE MEDICINA FAMILIAR                           1
HOSPITAL DE GINECO-PEDIATRÍA                          1
HOSPITAL GENERAL REGIONAL                             1
CONSULTA EXTERNA DE ESPECIALIDADES                    1
Name: count, dtype: int64

Quién administra los consultorios junto a farmacia.

In [30]:
op.loc[
    op["NOMBRE DE TIPOLOGIA"] == "CONSULTORIO ADYACENTE A FARMACIA", "NOMBRE DE LA INS ADM"
].value_counts()

NOMBRE DE LA INS ADM
FUNDACION BEST A. C.    201
BENAVIDES                69
NO ESPECIFICADO           6
OTRA                      1
Name: count, dtype: int64

Frecuencia por nivel de atención.

In [31]:
barras(conteo(op["NIVEL ATENCION"]), "Establecimientos en operación por nivel de atención")

Hallazgo: 777 de primer nivel, 103 de segundo, 6 de tercero y 57 sin nivel (NO APLICA); no hay vacíos.

Frecuencia por estrato (urbano o rural).

In [32]:
barras(conteo(op["ESTRATO UNIDAD"]), "Establecimientos en operación por estrato")

Hallazgo: 718 urbanos y 225 rurales, sin vacíos.

Frecuencia por municipio, con los 72 municipios.

In [33]:
barras(conteo(op["MUNICIPIO"]), "Establecimientos en operación por municipio")

Hallazgo: los 72 municipios tienen al menos un establecimiento en operación; Hermosillo tiene 213 y 12 municipios tienen solo uno.

Resumen:
- Los 277 consultorios junto a farmacia son todos privados y de primer nivel: casi 8 de cada 10 privados y más de un tercio del primer nivel.
- En segundo nivel, 52 de 103 tienen tipología "NO ESPECIFICADO", y 49 de ellos son privados.

## 6. Coordenadas

Clasificación de los registros: sin coordenadas, en (0, 0) o fuera de una caja aproximada que encierra a Sonora (no es el límite real del estado).

In [34]:
LAT_MIN, LAT_MAX = 26.0, 32.6
LON_MIN, LON_MAX = -115.2, -108.3

lat, lon = son["LATITUD"], son["LONGITUD"]
coord = pd.Series("dentro de la caja", index=son.index, name="COORDENADAS")
coord.loc[~(lat.between(LAT_MIN, LAT_MAX) & lon.between(LON_MIN, LON_MAX))] = "fuera de la caja"
coord.loc[lat.eq(0) & lon.eq(0)] = "en (0, 0)"
coord.loc[lat.isna() | lon.isna()] = "sin coordenadas"

pd.crosstab(coord, son["ESTATUS DE OPERACION"], margins=True, margins_name="Total")

ESTATUS DE OPERACION,EN OPERACION,FUERA DE OPERACION,PENDIENTE DE ENTRAR EN OPERACION,Total
COORDENADAS,,,,
dentro de la caja,876,717,4,1597
"en (0, 0)",0,1,0,1
fuera de la caja,56,0,0,56
sin coordenadas,11,18,0,29
Total,943,736,4,1683


Establecimientos fuera de la caja y dónde caen.

In [35]:
son[coord == "fuera de la caja"].groupby("NOMBRE COMERCIAL").agg(
    registros=("CLUES", "size"),
    municipios=("MUNICIPIO", "nunique"),
    lat_min=("LATITUD", "min"),
    lat_max=("LATITUD", "max"),
    lon_min=("LONGITUD", "min"),
    lon_max=("LONGITUD", "max"),
)

,registros,municipios,lat_min,lat_max,lon_min,lon_max
NOMBRE COMERCIAL,,,,,,
FARMACIAS BENAVIDES,1,1,31.7391,31.73910,-106.48670,-106.4867
FARMACIAS SIMILARES,55,11,19.2328,19.46312,-99.18886,-99.1194


Distribución de la latitud; el cursor sobre un punto muestra su CLUES y municipio.

In [36]:
caja("LATITUD", "Latitud")

Hallazgo: la mayoría está entre 27 y 30; abajo quedan 55 puntos cerca de 19 (Ciudad de México) y uno en 0.

Distribución de la longitud.

In [37]:
caja("LONGITUD", "Longitud")

Hallazgo: arriba quedan 55 puntos cerca de −99 (Ciudad de México), uno en −106.5 y uno en 0; los atípicos de abajo, cerca de −115, son reales (San Luis Río Colorado y Puerto Peñasco).

Mapa de los establecimientos con coordenadas, coloreados por estatus y centrado en México para ver también los puntos fuera de Sonora.

In [38]:
fig = px.scatter_map(
    son.dropna(subset=["LATITUD", "LONGITUD"]),
    lat="LATITUD",
    lon="LONGITUD",
    color="ESTATUS DE OPERACION",
    color_discrete_map=COLORES_ESTATUS,
    category_orders={"ESTATUS DE OPERACION": list(COLORES_ESTATUS)},
    hover_name="NOMBRE DE LA UNIDAD",
    hover_data=["CLUES", "MUNICIPIO", "NIVEL ATENCION"],
    center={"lat": 24, "lon": -106},
    zoom=4.5,
    map_style="carto-positron",
    height=600,
    title="Establecimientos de Sonora por estatus",
)
fig.update_traces(marker_size=8)
fig.show()

Hallazgo: casi todos los puntos caen en Sonora; fuera se ven el grupo de la Ciudad de México y un punto cerca de Ciudad Juárez, y el de (0, 0) queda fuera de la vista.

Número de decimales de las latitudes: con 1 o 2 decimales el punto puede quedar a kilómetros del lugar real.

In [39]:
decimales = son["LATITUD"].dropna().astype(str).str.split(".").str[1].str.len()
decimales.value_counts().sort_index()

LATITUD
1      2
2      7
3     58
4    668
5    201
6    483
7    235
Name: count, dtype: int64

Conteo de establecimientos en operación que comparten exactamente las mismas coordenadas.

In [40]:
op_coord = op.dropna(subset=["LATITUD", "LONGITUD"])
op_coord.duplicated(subset=["LATITUD", "LONGITUD"], keep=False).sum()

np.int64(30)

Hallazgos:
- 56 en operación fuera de Sonora, todos consultorios junto a farmacia: 55 de Farmacias Similares en la Ciudad de México y 1 de Farmacias Benavides de Huatabampo cerca de Ciudad Juárez.
- Con otros 11 sin coordenadas, solo 876 de 943 pasan esta revisión básica.
- 30 en operación comparten coordenadas exactas con otro establecimiento.

## 7. Código postal, fechas y unidades móviles

Distribución del código postal.

In [41]:
caja("CODIGO POSTAL", "Código postal")

Hallazgo: casi todos están entre 83000 y 85999, el rango de Sonora; 245 tienen 99999 y uno de Agua Prieta tiene 82100, un código de Sinaloa.

Distribución de los años de construcción, inicio de operación y baja, en la misma escala.

In [42]:
fechas = {
    "Construcción": "FECHA DE CONSTRUCCION",
    "Inicio de operación": "FECHA DE INICIO DE OPERACION",
    "Baja": "FECHA EFECTIVA DE BAJA",
}
anios = (
    pd.DataFrame(
        {nombre: pd.to_datetime(son[col], format="%d/%m/%Y").dt.year for nombre, col in fechas.items()}
    )
    .assign(CLUES=son["CLUES"])
    .melt(id_vars="CLUES", var_name="Fecha", value_name="Año")
    .dropna()
)
fig = px.box(anios, x="Fecha", y="Año", points="outliers", hover_data=["CLUES"], title="Años de las fechas")
fig.update_traces(marker_color=AZUL)
fig.update_layout(template="plotly_white", height=420, xaxis_title=None)
fig.show()

Hallazgo: construcción va de 1943 a 2025 e inicio de operación de 1948 a 2026, sin años imposibles; las bajas se juntan en 2020 (151) y 2024 (275).

Distribución del modelo (año) de las unidades móviles.

In [43]:
caja("UNIDAD MOVIL MODELO", "Modelo de unidad móvil")

Hallazgo: solo 30 registros tienen modelo, de 2005 a 2021, más un 9999 que es valor de relleno.

## 8. Texto libre

Palabras más frecuentes en OBSERVACIONES AL REGISTRO, contadas una vez por registro.

In [44]:
barras(
    palabras_por_registro(son["OBSERVACIONES AL REGISTRO"]).head(20),
    "Palabras más frecuentes en OBSERVACIONES AL REGISTRO",
    "Registros donde aparece",
)

Hallazgo: los textos vienen de plantillas; dominan las bajas aprobadas con documentos de la Dirección General de Planeación y Desarrollo y los traspasos a IMSS-Bienestar, y "ttransfiere" (con doble t) aparece en 272 registros.

Palabras más frecuentes en COMENTARIOS DE LA VALIDACION, contadas una vez por registro.

In [45]:
barras(
    palabras_por_registro(son["COMENTARIOS DE LA VALIDACION"]).head(20),
    "Palabras más frecuentes en COMENTARIOS DE LA VALIDACION",
    "Registros donde aparece",
)

Hallazgo: dominan los traspasos a IMSS-Bienestar (más de 500 registros) y las altas por carga masiva de Fundación Best (unos 140).

## 9. Notas para el diccionario

| Columna | Cómo viene | Nota |
|---|---|---|
| CLUES | texto de 11 caracteres | Llave única. 5 letras (entidad e institución) y 6 dígitos. En Sonora empieza con SR. |
| CLAVE DE LA ENTIDAD, CLAVE DEL MUNICIPIO, CLAVE DE LA LOCALIDAD | entero | Sin ceros a la izquierda. Para unir con INEGI hay que rellenar a 2, 3 y 4 dígitos. |
| CLAVE ESTATUS DE OPERACION, ESTATUS DE OPERACION | entero, texto | 1 en operación, 2 pendiente de entrar en operación, 3 fuera de operación, 4 en proceso de construcción. |
| CLAVE NIVEL ATENCION, NIVEL ATENCION | entero, texto | 1 primer nivel (consulta externa básica y prevención), 2 segundo (hospitales generales y clínicas de especialidades), 3 tercero (alta especialidad), 6 no aplica (sin atención médica directa: oficinas, almacenes, laboratorios, asistencia social). |
| NOMBRE DE LA INSTITUCION, NOMBRE DE LA INS ADM, NOMBRE DE TIPOLOGIA | texto | Algunos valores traen espacios al final y los acentos no son uniformes (CLINICA y CLÍNICA). "NO ESPECIFICADO" funciona como nulo. |
| LATITUD, LONGITUD | decimal | Grados decimales. En Sonora hay 29 sin dato, 1 en (0, 0) y 56 fuera del estado. Traen de 1 a 7 decimales. |
| CODIGO POSTAL | decimal | Conviene leerlo como texto de 5 dígitos. 99999 significa sin dato. |
| Columnas FECHA | texto | Formato dd/mm/aaaa. |
| VIALIDAD, NUMERO EXTERIOR, ASENTAMIENTO | texto | "SIN NOMBRE", "SIN NÚMERO" y los blancos funcionan como nulos. |
| OBSERVACIONES AL REGISTRO, COMENTARIOS DE LA VALIDACION | texto libre | Explican altas, bajas y traspasos, por ejemplo de la Secretaría de Salud estatal a IMSS-Bienestar. Pueden servir como dato de texto. |
| RFC y teléfonos | texto | No hacen falta para el análisis. |
| Columnas UNIDAD MOVIL | varios | Casi vacías (más de 96 % nulos) porque solo aplican a unidades móviles. |

El nombre del archivo dice 202608, pero los movimientos llegan al 22/09/2026. Hay que anotarlo en la descripción de la fuente.

Preguntas para el equipo:

- ¿Los 277 consultorios junto a farmacia cuentan como primer nivel?
- ¿Qué hacer con las 14 unidades móviles en operación? Su ubicación puede cambiar.
- ¿Corregir o dejar fuera los 56 establecimientos fuera de Sonora y los 11 sin coordenadas?
- ¿Los 8 establecimientos de consulta externa marcados como segundo nivel cuentan como segundo nivel?